In [33]:
# Cell 1: repo bootstrap + shared runtime setup.
import hashlib
import importlib
import json
import os
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "browser").exists():
    ROOT = ROOT.parent.resolve()
if not (ROOT / "browser").exists():
    raise RuntimeError("Could not locate repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import browser.driver as browser_driver
import browser.linkedin as linkedin
import core.config as core_config
import storage.embedded_mongo as embedded_mongo
import tasks.candidate_scoring_task as candidate_scoring_task
import tasks.detail_scoring_task as detail_scoring_task
import tasks.onboarding_task as onboarding_task

browser_driver = importlib.reload(browser_driver)
linkedin = importlib.reload(linkedin)
core_config = importlib.reload(core_config)
embedded_mongo = importlib.reload(embedded_mongo)
candidate_scoring_task = importlib.reload(candidate_scoring_task)
detail_scoring_task = importlib.reload(detail_scoring_task)
onboarding_task = importlib.reload(onboarding_task)

from browser.driver import build_driver
from core.config import load_app_config

app_config = load_app_config()
browser_cfg = dict(app_config["profile"]["browser"])
browser_cfg["headless"] = False
browser_cfg["start_maximized"] = True

old_user_data_dir = Path(r"D:\_Desktop\Projects\Automations prj\User Data")
if old_user_data_dir.exists():
    browser_cfg["user_data_dir"] = str(old_user_data_dir)

store = embedded_mongo.EmbeddedMongoStore(Path(app_config["profile"]["mongo_file"]))
driver = build_driver(browser_cfg)

onboarding_state_path = ROOT / "runtime" / "task_states" / "onboarding_task_state.json"
candidate_scoring_state_path = ROOT / "runtime" / "task_states" / "candidate_scoring_task_state.json"
detail_scoring_state_path = ROOT / "runtime" / "task_states" / "detail_scoring_task_state.json"
artifact_root = ROOT / "runtime" / "notebook_artifacts" / "candidate_pipeline"
candidate_artifact_dir = artifact_root / "candidates"
detail_artifact_dir = artifact_root / "details"
candidate_artifact_dir.mkdir(parents=True, exist_ok=True)
detail_artifact_dir.mkdir(parents=True, exist_ok=True)


def _save_json(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)


def _load_json(path: Path) -> dict[str, object]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _canonicalize(value):
    if isinstance(value, dict):
        return {str(key): _canonicalize(value[key]) for key in sorted(value)}
    if isinstance(value, list):
        return [_canonicalize(item) for item in value]
    if isinstance(value, tuple):
        return [_canonicalize(item) for item in value]
    return value


def _parse_pages_spec(pages):
    if pages is None or pages == "":
        return [1]
    if isinstance(pages, int):
        return [pages] if pages > 0 else [1]
    if isinstance(pages, list):
        result = []
        for item in pages:
            if isinstance(item, int) and item > 0 and item not in result:
                result.append(item)
            elif isinstance(item, str):
                for page in _parse_pages_spec(item):
                    if page not in result:
                        result.append(page)
        return result or [1]
    if isinstance(pages, str):
        text = pages.strip()
        if not text:
            return [1]
        result = []
        for chunk in text.replace(" ", ",").split(","):
            piece = chunk.strip()
            if not piece:
                continue
            if "-" in piece:
                left, right = [part.strip() for part in piece.split("-", 1)]
                if left.isdigit() and right.isdigit():
                    start = int(left)
                    end = int(right)
                    if start > end:
                        start, end = end, start
                    for page in range(start, end + 1):
                        if page > 0 and page not in result:
                            result.append(page)
                    continue
            if piece.isdigit():
                page = int(piece)
                if page > 0 and page not in result:
                    result.append(page)
        return result or [1]
    try:
        page = int(str(pages).strip())
        return [page] if page > 0 else [1]
    except Exception:
        return [1]


def candidate_search_signature(candidate_search_input: dict[str, object]) -> str:
    normalized = {
        "keyword": candidate_search_input.get("keyword", ""),
        "location": candidate_search_input.get("location", ""),
        "filters": _canonicalize(candidate_search_input.get("filters", {})),
        "pages": _parse_pages_spec(candidate_search_input.get("pages")),
    }
    raw = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


def candidate_artifact_path(candidate_search_input: dict[str, object]) -> Path:
    return candidate_artifact_dir / f"{candidate_search_signature(candidate_search_input)}.json"


def detail_artifact_path(listing_id: str) -> Path:
    return detail_artifact_dir / f"{str(listing_id).strip()}.json"


def load_or_fetch_candidate_result(candidate_search_input: dict[str, object]) -> tuple[dict[str, object], bool, Path]:
    path = candidate_artifact_path(candidate_search_input)
    if path.exists():
        payload = _load_json(path)
        return payload.get("candidate_result", {}), True, path
    candidate_result = linkedin.fetch_job_listings(
        driver,
        keyword=candidate_search_input["keyword"],
        location=candidate_search_input["location"],
        filters=candidate_search_input.get("filters"),
        pages=candidate_search_input.get("pages"),
        delays=candidate_search_input.get("delays"),
        verbose=bool(candidate_search_input.get("verbose", False)),
    )
    _save_json(
        path,
        {
            "candidate_search_input": candidate_search_input,
            "candidate_result": candidate_result,
        },
    )
    return candidate_result, False, path


def load_or_fetch_detail_result(candidate_result: dict[str, object], listing_id: str, detail_fetch_input: dict[str, object]) -> tuple[dict[str, object], bool, Path]:
    path = detail_artifact_path(listing_id)
    if path.exists():
        payload = _load_json(path)
        return payload.get("detail_result", {}), True, path
    detail_result = linkedin.fetch_listings_description(
        driver,
        candidate_result,
        listing_id=str(listing_id),
        delays=detail_fetch_input.get("delays"),
        verbose=bool(detail_fetch_input.get("verbose", False)),
    )
    _save_json(
        path,
        {
            "listing_id": str(listing_id),
            "detail_fetch_input": detail_fetch_input,
            "detail_result": detail_result,
        },
    )
    return detail_result, False, path


runtime_state = {
    "root": ROOT,
    "driver": driver,
    "store": store,
    "browser_cfg": browser_cfg,
    "onboarding_state_path": onboarding_state_path,
    "candidate_scoring_state_path": candidate_scoring_state_path,
    "detail_scoring_state_path": detail_scoring_state_path,
    "candidate_artifact_dir": candidate_artifact_dir,
    "detail_artifact_dir": detail_artifact_dir,
    "onboarding_result": {},
    "candidate_result": {},
    "scoring_result": {},
    "detail_fetch_result": {},
    "detail_scoring_result": {},
    "candidate_search_input": {},
    "candidate_scoring_input": {},
    "detail_fetch_input": {},
    "detail_scoring_input": {},
}

print(json.dumps({
    "root": str(ROOT),
    "browser": browser_cfg,
    "onboarding_state_path": str(onboarding_state_path),
    "candidate_scoring_state_path": str(candidate_scoring_state_path),
    "detail_scoring_state_path": str(detail_scoring_state_path),
    "candidate_artifact_dir": str(candidate_artifact_dir),
    "detail_artifact_dir": str(detail_artifact_dir),
}, indent=2, ensure_ascii=False))


config: dir D:\_Desktop\Projects\Automations prj\Job_search\JobSeekr_fresh\config
config: load profile.json
config: load extract.json
config: load presets.json
config: load llm_backend.json
store: open runtime\mongo_store.json
driver: start
driver: launch chrome
driver: ready
{
  "root": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh",
  "browser": {
    "headless": false,
    "use_stealth": true,
    "version_main": 149,
    "page_load_timeout_seconds": 15,
    "wait_timeout_seconds": 20,
    "start_maximized": true,
    "user_data_dir": "D:\\_Desktop\\Projects\\Automations prj\\User Data"
  },
  "onboarding_state_path": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh\\runtime\\task_states\\onboarding_task_state.json",
  "candidate_scoring_state_path": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh\\runtime\\task_states\\candidate_scoring_task_state.json",
  "detail_scoring_state_path": "D:\\_Desktop\\Projects\\Automatio

In [34]:
# Cell 2: run onboarding digitization.
import importlib
import json
import tasks.onboarding_task as onboarding_task

onboarding_task = importlib.reload(onboarding_task)

profile_number = "1"
profiles_root = ROOT / "profiles"
selected_profile_dir = profiles_root / profile_number

onboarding_input = {
    "task_name": "onboarding_profile_digitization",
    "task_id": "onboarding-stage-001",
    "documents": [str(selected_profile_dir)],
}

onboarding_result = onboarding_task.run_onboarding_task(
    onboarding_input,
    state_path=runtime_state["onboarding_state_path"],
    heartbeat_seconds=1.0,
    step_delay_seconds=0.5,
    verbose=False,
)

runtime_state["onboarding_result"] = onboarding_result

print(json.dumps({
    "status": onboarding_result.get("status"),
    "missing_fields": onboarding_result.get("missing_fields", []),
    "ready_for_scoring": onboarding_result.get("completeness", {}).get("ready_for_scoring"),
    "confidence_score": onboarding_result.get("completeness", {}).get("confidence_score"),
    "full_name": onboarding_result.get("digitized_user", {}).get("identity", {}).get("full_name"),
    "location": onboarding_result.get("digitized_user", {}).get("contact", {}).get("location"),
    "roles": onboarding_result.get("digitized_user", {}).get("preferences", {}).get("roles", []),
}, indent=2, ensure_ascii=False))


{
  "status": "success",
  "missing_fields": [],
  "ready_for_scoring": true,
  "confidence_score": 100,
  "full_name": "Omar Saeed Al Shamsi",
  "location": "Dubai, United Arab Emirates",
  "roles": [
    "Data Engineer",
    "Data Analyst",
    "Analytics Engineer",
    "Business Intelligence Analyst",
    "Junior Machine Learning Engineer",
    "AI/Data Solutions Engineer",
    "Treat this as an entry-level / recent-graduate profile"
  ]
}


In [63]:
# Cell 3: fetch candidate listings with the LinkedIn search tool.
import importlib
import json
import browser.linkedin as linkedin

linkedin = importlib.reload(linkedin)

candidate_search_location = onboarding_result.get("digitized_user", {}).get("contact", {}).get("location") or "United States"

candidate_search_input = {
    "keyword": "bIG DatA",
    "location": candidate_search_location,
    "filters": {
        "experience_level": "Entry level",
        "date_posted": "Past 24 hours",
        "job_type": "Full-time"
    },
    "pages": "1",
    "delays": {
        "open_jobs_search_page": 1,
        "set_keyword_input": 1,
        "set_location_input": 1,
        "open_all_filters_menu": 1,
        "sync_filters_state": 0.2,
        "show_results": 1,
        "go_to_page": 3,
        "click_listing_card": 1,
        "click_listing_card_jitter": 0.2,
    },
    "verbose": False,
}

candidate_result, reused_candidate_artifact, candidate_artifact_file = load_or_fetch_candidate_result(candidate_search_input)

runtime_state["candidate_search_input"] = candidate_search_input
runtime_state["candidate_result"] = candidate_result

print(json.dumps({
    "status": candidate_result.get("status"),
    "pages_requested": candidate_result.get("pages_requested", []),
    "listing_count": len(candidate_result.get("listings", [])),
    "search_task_id": candidate_result.get("search_task", {}).get("id", ""),
    "pages_fetched": candidate_result.get("search_task", {}).get("pages_fetched", []),
    "visible_unfetched_pages": candidate_result.get("search_task", {}).get("visible_unfetched_pages", []),
    "artifact_reused": reused_candidate_artifact,
    "artifact_path": str(candidate_artifact_file),
    "location_used": candidate_search_location,
}, indent=2, ensure_ascii=False))

print(json.dumps(candidate_result.get("listings", []), indent=2, ensure_ascii=False))


{
  "status": "success",
  "pages_requested": [
    1
  ],
  "listing_count": 7,
  "search_task_id": "e2001ef9da",
  "pages_fetched": [
    1
  ],
  "visible_unfetched_pages": [],
  "artifact_reused": false,
  "artifact_path": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh\\runtime\\notebook_artifacts\\candidate_pipeline\\candidates\\2bb8e9ce40199ca7.json",
  "location_used": "Dubai, United Arab Emirates"
}
[
  {
    "title": "Typist",
    "company": "Undercover CX Consultancy",
    "location": "Dubai, United Arab Emirates (On-site)",
    "link": "https://www.linkedin.com/jobs/view/4433786871/?eBP=NOT_ELIGIBLE_FOR_CHARGING&refId=wqzqhvGhASfbwLCKmwRF%2Bw%3D%3D&trackingId=m43Zy21AAG3saanyZABI9w%3D%3D&trk=flagship3_search_srp_jobs",
    "job_id": "4433786871",
    "promoted": false,
    "easy_apply": true,
    "listed_on": null
  },
  {
    "title": "Specialist (UAE Nationals Only) with verification",
    "company": "Standard Chartered",
    "location": "Dubai, Dubai

In [51]:
# Cell 4: score candidate listings for the next stage.
import importlib
import json
import tasks.candidate_scoring_task as candidate_scoring_task

candidate_scoring_task = importlib.reload(candidate_scoring_task)

candidate_scoring_input = {
    "task_name": "candidate_listing_scoring",
    "task_id": "candidate-scoring-stage-001",
    "digitized_user": onboarding_result.get("digitized_user", {}),
    "candidates": candidate_result.get("listings", []),
    "batch_size": 15,
}

scoring_result = candidate_scoring_task.run_candidate_scoring_task(
    candidate_scoring_input,
    state_path=runtime_state["candidate_scoring_state_path"],
    heartbeat_seconds=1.0,
    step_delay_seconds=0.2,
    verbose=False,
)

runtime_state["candidate_scoring_input"] = candidate_scoring_input
runtime_state["scoring_result"] = scoring_result

preview = [
    {
        "company": row.get("company"),
        "listing_id": row.get("listing_id"),
        "reason": row.get("exclude_reason_code"),
    }
    for row in scoring_result.get("excluded_candidates", [])
]

print(json.dumps({
    "status": scoring_result.get("status"),
    "total_candidates": scoring_result.get("summary", {}).get("total_candidates"),
    "batch_count": scoring_result.get("summary", {}).get("batch_count"),
    "kept_count": scoring_result.get("summary", {}).get("kept_count"),
    "excluded_count": scoring_result.get("summary", {}).get("excluded_count"),
    "reason_histogram": scoring_result.get("summary", {}).get("reason_histogram", {}),
    "next_stage_candidate_count": len(scoring_result.get("next_stage_candidates", [])),
    "preview": preview,
}, indent=2, ensure_ascii=False))

print(json.dumps([
    {
        "batch_index": batch.get("batch_index"),
        "llm_response": batch.get("llm_response"),
        "llm_error": batch.get("llm_error"),
    }
    for batch in scoring_result.get("batches", [])
], indent=2, ensure_ascii=False))

print(json.dumps(scoring_result.get("excluded_candidates", []), indent=2, ensure_ascii=False))


{
  "status": "success",
  "total_candidates": 3,
  "batch_count": 1,
  "kept_count": 0,
  "excluded_count": 3,
  "reason_histogram": {
    "constraint_conflict": 3
  },
  "next_stage_candidate_count": 0,
  "preview": [
    {
      "company": "Nestlé",
      "listing_id": "4436998383",
      "reason": "constraint_conflict"
    },
    {
      "company": "Nestlé",
      "listing_id": "4436993530",
      "reason": "constraint_conflict"
    },
    {
      "company": "TikTok",
      "listing_id": "4345579332",
      "reason": "constraint_conflict"
    }
  ]
}
[
  {
    "batch_index": 1,
    "llm_response": "{\"excluded\": [{\"company\": \"Nestlé\", \"listing_id\": \"4436998383\", \"reason\": \"Internship position conflicts with full-time employment hard_no constraint\"}, {\"company\": \"Nestlé\", \"listing_id\": \"4436993530\", \"reason\": \"Internship position conflicts with full-time employment hard_no constraint\"}, {\"company\": \"TikTok\", \"listing_id\": \"4345579332\", \"reason\": \"

In [41]:
# Cell 5: fetch full detail rows for every non-excluded candidate, reusing notebook artifacts when available.
import importlib
import json
import browser.linkedin as linkedin

linkedin = importlib.reload(linkedin)

non_excluded_candidates = scoring_result.get("next_stage_candidates", [])

detail_fetch_input = {
    "delays": {
        "click_listing_card": 13,
        "click_listing_card_jitter": 4,
    },
    "verbose": False,
}

fetched_detail_rows = []
detail_fetch_artifacts = []
artifact_reuse_count = 0
for candidate in non_excluded_candidates:
    listing_id = str(candidate.get("listing_id") or candidate.get("job_id") or "").strip()
    if not listing_id:
        continue
    detail_result, reused_detail_artifact, detail_artifact_file = load_or_fetch_detail_result(
        candidate_result,
        listing_id,
        detail_fetch_input,
    )
    artifact_reuse_count += int(bool(reused_detail_artifact))
    detail_fetch_artifacts.append({
        "listing_id": listing_id,
        "artifact_reused": reused_detail_artifact,
        "artifact_path": str(detail_artifact_file),
    })
    for item in detail_result.get("ai", []):
        merged_row = {
            "listing": item.get("listing", {}),
            "detail": item.get("detail", {}),
            "company_profile": item.get("company_profile", {}),
            "dev": detail_result.get("dev", {}),
        }
        merged_row["listing_id"] = listing_id
        fetched_detail_rows.append(merged_row)

runtime_state["detail_fetch_input"] = detail_fetch_input
runtime_state["detail_fetch_result"] = {
    "rows": fetched_detail_rows,
    "artifacts": detail_fetch_artifacts,
    "artifact_reuse_count": artifact_reuse_count,
}

print(json.dumps({
    "candidate_count": len(non_excluded_candidates),
    "detail_row_count": len(fetched_detail_rows),
    "artifact_reuse_count": artifact_reuse_count,
    "artifacts": detail_fetch_artifacts,
}, indent=2, ensure_ascii=False))

print(json.dumps(fetched_detail_rows, indent=2, ensure_ascii=False))


{
  "candidate_count": 4,
  "detail_row_count": 4,
  "artifact_reuse_count": 0,
  "artifacts": [
    {
      "listing_id": "4436018906",
      "artifact_reused": false,
      "artifact_path": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh\\runtime\\notebook_artifacts\\candidate_pipeline\\details\\4436018906.json"
    },
    {
      "listing_id": "4436018010",
      "artifact_reused": false,
      "artifact_path": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh\\runtime\\notebook_artifacts\\candidate_pipeline\\details\\4436018010.json"
    },
    {
      "listing_id": "4400554163",
      "artifact_reused": false,
      "artifact_path": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh\\runtime\\notebook_artifacts\\candidate_pipeline\\details\\4400554163.json"
    },
    {
      "listing_id": "4433794293",
      "artifact_reused": false,
      "artifact_path": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fres

In [44]:
# Cell 6: score the fetched job details.
import importlib
import json
import tasks.detail_scoring_task as detail_scoring_task

detail_scoring_task = importlib.reload(detail_scoring_task)

detail_scoring_input = {
    "task_name": "detail_listing_scoring",
    "task_id": "detail-scoring-stage-001",
    "digitized_user": onboarding_result.get("digitized_user", {}),
    "detail_rows": fetched_detail_rows,
    "batch_size": 10,
}

detail_scoring_result = detail_scoring_task.run_detail_scoring_task(
    detail_scoring_input,
    state_path=runtime_state["detail_scoring_state_path"],
    heartbeat_seconds=1.0,
    step_delay_seconds=0.2,
    verbose=False,
)

runtime_state["detail_scoring_input"] = detail_scoring_input
runtime_state["detail_scoring_result"] = detail_scoring_result

print(json.dumps({
    "status": detail_scoring_result.get("status"),
    "total_rows": detail_scoring_result.get("summary", {}).get("total_rows"),
    "batch_count": detail_scoring_result.get("summary", {}).get("batch_count"),
    "kept_count": detail_scoring_result.get("summary", {}).get("kept_count"),
    "excluded_count": detail_scoring_result.get("summary", {}).get("excluded_count"),
    "reason_histogram": detail_scoring_result.get("summary", {}).get("reason_histogram", {}),
    "section_histogram": detail_scoring_result.get("summary", {}).get("section_histogram", {}),
    "next_stage_row_count": len(detail_scoring_result.get("next_stage_rows", [])),
}, indent=2, ensure_ascii=False))

print(json.dumps([
    {
        "batch_index": batch.get("batch_index"),
        "llm_response": batch.get("llm_response"),
        "llm_error": batch.get("llm_error"),
    }
    for batch in detail_scoring_result.get("batches", [])
], indent=2, ensure_ascii=False))

print(json.dumps(detail_scoring_result.get("excluded_detail_rows", []), indent=2, ensure_ascii=False))
print(json.dumps(detail_scoring_result.get("kept_detail_rows", []), indent=2, ensure_ascii=False))
print(json.dumps(detail_scoring_result.get("scored_detail_rows", []), indent=2, ensure_ascii=False))


{
  "status": "success",
  "total_rows": 4,
  "batch_count": 1,
  "kept_count": 0,
  "excluded_count": 4,
  "reason_histogram": {
    "detail_conflict": 4
  },
  "section_histogram": {
    "compensation": {
      "yes": 1,
      "partial": 3,
      "no": 0
    },
    "progression": {
      "yes": 0,
      "partial": 1,
      "no": 3
    },
    "work_style": {
      "yes": 0,
      "partial": 4,
      "no": 0
    },
    "relevance": {
      "yes": 0,
      "partial": 0,
      "no": 4
    },
    "company_signal": {
      "yes": 1,
      "partial": 3,
      "no": 0
    },
    "risks": {
      "yes": 4,
      "partial": 0,
      "no": 0
    }
  },
  "next_stage_row_count": 0
}
[
  {
    "batch_index": 1,
    "llm_response": "```json\n{\n  \"excluded\": [\n    {\n      \"company\": \"Sanvi Engineering And Business Consulting Solutions\",\n      \"listing_id\": \"4436018906\",\n      \"reason\": \"Requires Mechanical/Automobile engineering degree and Arabic fluency for sales role; salary AED